#**Análisis de Datos – Ciencia de Datos**

# Análisis de Datos – Ciencia de Datos
### Comisión 25262 – Credor: Juan Rojas

Este notebook integra las etapas de recopilación, preparación, limpieza, transformación y análisis de datos, aplicando herramientas de Python y Pandas.  
Se utilizan tres datasets: **ventas.csv**, **clientes.csv** y **marketing.csv**, ubicados en Google Drive.

---


# **Etapa 1: Recopilación y Preparación de Datos**

In [ ]:
import pandas as pd
import numpy as np

# Definición de rutas
ruta_ventas = "/content/drive/MyDrive/datasets/ventas.csv"
ruta_clientes = "/content/drive/MyDrive/datasets/clientes.csv"
ruta_marketing = "/content/drive/MyDrive/datasets/marketing.csv"

# Carga de datasets
ventas = pd.read_csv(ruta_ventas)
clientes = pd.read_csv(ruta_clientes)
marketing = pd.read_csv(ruta_marketing)

# Validación inicial
print("ventas.shape ->", ventas.shape)
print("clientes.shape ->", clientes.shape)
print("marketing.shape ->", marketing.shape)

display(ventas.head(3))
display(clientes.head(3))
display(marketing.head(3))


**# Exploración inicial (EDA básico)**

In [ ]:
def eda(df, nombre):
    print(f"=== {nombre} ===")
    print("Dimensiones:", df.shape)
    print("Columnas:", list(df.columns))
    print("Tipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isna().sum())
    print("\nPrimeras filas:")
    display(df.head(5))
    print("\nEstadísticas descriptivas:")
    display(df.describe(include='number'))
    print("-"*100)

eda(ventas, "VENTAS (inicial)")
eda(clientes, "CLIENTES (inicial)")
eda(marketing, "MARKETING (inicial)")


# **Etapa 2: Limpieza y Normalización**

In [ ]:
# Copias limpias
ventas_clean = ventas.drop_duplicates().copy()
clientes_clean = clientes.drop_duplicates().copy()
marketing_clean = marketing.drop_duplicates().copy()

# Normalización de texto
def normalizar_texto(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = (
            df[col].astype(str)
            .str.strip()
            .str.replace(r"[\u200b\t\r\n]", "", regex=True)
            .str.replace(" +", " ", regex=True)
            .str.title()
        )
    return df

ventas_clean = normalizar_texto(ventas_clean)
clientes_clean = normalizar_texto(clientes_clean)
marketing_clean = normalizar_texto(marketing_clean)

# Normalización de fechas
for df in [ventas_clean, clientes_clean, marketing_clean]:
    for col in df.columns:
        if "fecha" in col.lower():
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

# Normalización de numéricos
ventas_clean["precio"] = (
    ventas_clean["precio"].astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
ventas_clean["precio"] = pd.to_numeric(ventas_clean["precio"], errors="coerce")

ventas_clean["cantidad"] = pd.to_numeric(ventas_clean["cantidad"], errors="coerce").astype("Int64")


# **Reporte Global de Calidad**

In [ ]:
def reporte_calidad_global(dfs, nombres):
    resumen = []
    for df, nombre in zip(dfs, nombres):
        resumen.append({
            "Dataset": nombre,
            "Filas": len(df),
            "Columnas": len(df.columns),
            "Nulos totales": df.isna().sum().sum(),
            "Duplicados": df.duplicated(keep=False).sum()
        })
    return pd.DataFrame(resumen)

display(reporte_calidad_global([ventas, clientes, marketing],
                               ["VENTAS Original", "CLIENTES Original", "MARKETING Original"]))
display(reporte_calidad_global([ventas_clean, clientes_clean, marketing_clean],
                               ["VENTAS Limpio", "CLIENTES Limpio", "MARKETING Limpio"]))


**# Etapa 2: Transformación y Agregación**

In [ ]:
# Ingreso por registro
ventas_perf = ventas_clean.assign(
    ingreso = ventas_clean["precio"] * ventas_clean["cantidad"]
)

# Resumen por producto
resumen_prod = (
    ventas_perf.groupby("producto", as_index=False)
    .agg(
        ingreso_total=('ingreso', 'sum'),
        unidades=('cantidad', 'sum'),
        precio_promedio=('precio', 'mean'),
        registros=('ingreso', 'size')
    )
    .sort_values(by="ingreso_total", ascending=False)
)
resumen_prod["precio_promedio"] = resumen_prod["precio_promedio"].round(2)

# Percentil 80
p80_ingreso = resumen_prod["ingreso_total"].quantile(0.80)
ventas_top = resumen_prod.query("ingreso_total >= @p80_ingreso").sort_values(
    by=["ingreso_total", "unidades"], ascending=[False, False]
)

print(f"Umbral P80: {p80_ingreso:,.2f}")
display(ventas_top.head(20))


# **Resumen por Categoría**

In [ ]:
resumen_cat = (
    ventas_perf.groupby("categoria", as_index=False)
    .agg(
        ingreso_total=('ingreso', 'sum'),
        unidades=('cantidad', 'sum'),
        ventas=('ingreso', 'size'),
        precio_promedio=('precio', 'mean')
    )
    .sort_values(by="ingreso_total", ascending=False)
)
resumen_cat["ticket_promedio_por_venta"] = resumen_cat["ingreso_total"] / resumen_cat["ventas"]

display(resumen_cat.head(20))


# **Integración Ventas + Marketing**

In [ ]:
df_ventas_x_producto = ventas_clean.groupby('producto').agg(
    precio_por_cantidad=('precio', lambda x: (x * ventas_clean.loc[x.index, 'cantidad']).sum())
).reset_index()

df_marketing_x_producto = marketing_clean.groupby('producto')['costo'].sum().reset_index()

merged_df = pd.merge(df_ventas_x_producto, df_marketing_x_producto, on='producto')
merged_df['porcentaje_costo_sobre_ingreso'] = (merged_df['costo'] / merged_df['precio_por_cantidad']) * 100

display(merged_df)


# **Visualización de eficiencia de campañas**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.boxplot(y=merged_df['porcentaje_costo_sobre_ingreso'])
plt.title("Boxplot del Porcentaje de Costo sobre Ingreso")
plt.show()

plt.figure(figsize=(8,6))
sns.histplot(merged_df['porcentaje_costo_sobre_ingreso'], kde=True)
plt.title("Distribución del Porcentaje de Costo sobre Ingreso")
plt.show()


# **Etapa 3: Estadística Descriptiva**

In [ ]:
media = resumen_prod['ingreso_total'].mean()
mediana = resumen_prod['ingreso_total'].median()
moda = resumen_prod['ingreso_total'].mode()
rango = resumen_prod['ingreso_total'].max() - resumen_prod['ingreso_total'].min()
varianza = resumen_prod['ingreso_total'].var(ddof=1)
desviacion = resumen_prod['ingreso_total'].std(ddof=1)

print(f"Media: {media:,.2f}")
print(f"Mediana: {mediana:,.2f}")
print("Moda:", moda.tolist())
print(f"Rango: {rango:,.2f}")
print(f"Varianza: {varianza:,.2f}")
print(f"Desviación estándar: {desviacion:,.2f}")


# **Etapa 3.2: Análisis Exploratorio (EDA con Seaborn)**

In [ ]:
# Scatterplot precio vs cantidad
plt.figure(figsize=(8,6))
sns.scatterplot(data=ventas_perf, x="precio", y="cantidad", alpha=0.6)
plt.title("Relación Precio vs Cantidad")
plt.show()

# Heatmap de correlaciones
plt.figure(figsize=(8,6))
sns.heatmap(ventas_perf[['precio','cantidad','ingreso']].corr(), annot=True, cmap="Blues")
plt.title("Mapa de Correlaciones")
plt.show()

# Histograma de ingresos
plt.figure(figsize=(8,6))
sns.histplot(ventas_perf['ingreso'], bins=30, kde=True, color="skyblue")
plt.title("Distribución de Ingresos por Registro")
plt.show()


# **Tipos Gráficos para representar en Análisis de Datos**


1. Barras → compara ingresos por producto (Top 15).

2. Líneas → evolución temporal de ingresos.

3. Dispersión → relación entre precio y cantidad.

4. Histograma → distribución de ingresos individuales.

5. Boxplot → dispersión del porcentaje costo/ingreso.

6. Heatmap → correlaciones entre precio, cantidad e ingreso.

7. Pastel → participación de categorías en el ingreso total.

In [ ]:
# ============================================
# Tipos de Gráficos con Matplotlib y Seaborn
# ============================================

import matplotlib.pyplot as plt
import seaborn as sns

# 1️⃣ Gráfico de barras – Ingresos totales por producto
plt.figure(figsize=(12,6))
df_ordenado = resumen_prod.sort_values("ingreso_total", ascending=False).head(15)
sns.barplot(data=df_ordenado, x="producto", y="ingreso_total", color="skyblue")
plt.title("Ingresos Totales por Producto (Top 15)")
plt.xticks(rotation=90)
plt.show()

# 2️⃣ Gráfico de líneas – Evolución temporal de ventas
plt.figure(figsize=(12,6))
ventas_time = ventas_clean.groupby("fecha_venta")["ingreso"].sum().reset_index()
sns.lineplot(data=ventas_time, x="fecha_venta", y="ingreso", marker="o")
plt.title("Evolución Temporal de Ingresos")
plt.xlabel("Fecha de Venta")
plt.ylabel("Ingreso Total")
plt.show()

# 3️⃣ Gráfico de dispersión – Precio vs Cantidad
plt.figure(figsize=(8,6))
sns.scatterplot(data=ventas_perf, x="precio", y="cantidad", alpha=0.6)
plt.title("Relación Precio vs Cantidad")
plt.show()

# 4️⃣ Histograma – Distribución de ingresos por registro
plt.figure(figsize=(8,6))
sns.histplot(ventas_perf["ingreso"], bins=30, kde=True, color="skyblue")
plt.title("Distribución de Ingresos por Registro")
plt.xlabel("Ingreso")
plt.ylabel("Frecuencia")
plt.show()

# 5️⃣ Boxplot – Porcentaje de costo sobre ingreso (marketing vs ventas)
plt.figure(figsize=(8,6))
sns.boxplot(y=merged_df["porcentaje_costo_sobre_ingreso"], color="lightgreen")
plt.title("Boxplot del Porcentaje de Costo sobre Ingreso")
plt.ylabel("Porcentaje (%)")
plt.show()

# 6️⃣ Heatmap – Correlaciones entre variables numéricas
plt.figure(figsize=(8,6))
sns.heatmap(ventas_perf[["precio","cantidad","ingreso"]].corr(), annot=True, cmap="Blues")
plt.title("Mapa de Correlaciones")
plt.show()

# 7️⃣ Gráfico de pastel – Participación de categorías en ingresos
plt.figure(figsize=(8,6))
cat_share = resumen_cat.groupby("categoria")["ingreso_total"].sum()
plt.pie(cat_share, labels=cat_share.index, autopct="%1.1f%%", startangle=90, colors=sns.color_palette("pastel"))
plt.title("Participación de Categorías en Ingresos Totales")
plt.show()
